# Python程序设计（项目实战） · 工单P2.2《NumPy故障特征提取系统》
**第4周 | 2课时(80分钟) | 理实一体**  | 教师参考版

| 层 | 任务定位 | 达标要求 |
|----|---------|---------|
| **A 师傅** | 进阶拓展 | 相关系数矩阵热力图：剔除|r|>0.95的冗余特征并说明取舍；示范讲解 |
| **B 组长** | 标准达成 | <100ms+特征字典含物理含义 |
| **C 操作员** | 基础完成 | 30特征正确+向量化为主 |
| **D 学徒** | 跟做保底 | 跟做示例跑通第①个留白(var), 程序出结果 |
> 每4周按工单成绩动态升降层（连续2次≥B升一层）

## 一、课前资讯（Predict·预测）
> **预习快问**：x**2向量化为什么比for快100倍？

> **课前检查**：模板仓库已克隆更新；导学单与实施计划已提交平台

## 二、PRIMM五步法（先读后写）
本工单按国际编程教学法 **PRIMM** 组织：
1. **Predict 预测**——先运行下方示例，猜输出再验证；
2. **Run 运行**——跑通环境与数据；
3. **Investigate 研究**——读懂starter每段在做什么；
4. **Modify 修改**——把骨架留白处补全（教师版已逐处给出完成态）；
5. **Make 创造**——独立完成工单任务。


In [ ]:
# --- 环境与数据准备(勿改, 运行即可) ---
import os, sys, subprocess
os.chdir(os.path.abspath("../"))          # 切到课程包根目录(data/所在处)
print("工作目录:", os.getcwd())


## 三、任务要求（Make）
**实施步骤**
1. 跑通示例特征(mean/std/max/ptp/rms)
2. 补时域5个(var/crest/峭度/裕度)
3. spectrum()单边幅值谱

**验收标准**（工单《P2.2》「验收标准」四行原文）：`及格(D)`＝`30特征计算正确；向量化为主；标准化正确；PEP 8≥75%` ｜ `良好(C)`＝`1000样本<100ms；特征字典含物理含义` ｜ `优秀(B)`＝`<50ms；广播运用充分；箱线图输出；文档完整` ｜ `超优(A)`＝`优秀+衍生特征5个+相关性特征选择(去冗余)`
> **口径说明（别把两套字母混为一谈）**：上四行是**工单《P2.2》「验收标准」原文**（及格/良好/优秀/超优），与本课课堂分层（D 跟做保底 / C 基础完成 / B 标准达成 / A 进阶拓展）**不同轴**——课堂 C≈工单及格、课堂 B≈工单良好，工单「优秀」档（如 <50ms；广播运用充分；箱线图输出；文档完整）课堂阶梯不覆盖。工程化项目的 `output/report.txt` 已按工单原文逐行取证：p11/p21/p33/p55 直接以这四行为分层口径，p12/p22/p41/p66 另设「工单验收四档对照」节；本机取不了证的子项逐条点名，不假判。


### Starter 1：`# -*- coding: utf-8 -*-`
（教师版：本段已是**完成态**，可直接演示讲解；学生版对应处为留白。）

In [ ]:
# -*- coding: utf-8 -*-
"""P2.2 NumPy 30维故障特征提取 —— 教师版完成态(特征字典 + 30个特征全部实现)
运行: python W04_features.py          # 生成特征+箱线图+计时
数据: 自动合成振动信号(正常/轴承磨损两类, 100Hz采样, 固定随机种子42)
离线: 不读任何外部文件/不联网, 全部结果可复现
说明: 学生骨架的每个留白处, 本文件都给了实现, 并在实现上方标了 `# 教师版完成态`
"""
import time

import numpy as np

rng = np.random.RandomState(42)

### Starter 2：`def make_signals(n=200, length=1000, fs=100):`
（教师版：本段已是**完成态**，可直接演示讲解；学生版对应处为留白。）

In [ ]:
def make_signals(n=200, length=1000, fs=100):
    """合成两类信号: 正常(低幅+白噪) / 轴承磨损(冲击+高频成分)"""
    t = np.arange(length) / fs
    sigs, labels = [], []
    for i in range(n):
        base = 0.5 + 0.1 * np.sin(2 * np.pi * 8 * t)
        s = base + rng.normal(0, 0.05, length)
        if i >= n // 2:                      # 磨损: 加冲击与高频
            s += 1.2 * np.abs(np.sin(2 * np.pi * 33 * t)) * rng.binomial(1, 0.08, length)
            s += 0.4 * np.sin(2 * np.pi * 60 * t)
        sigs.append(s)
        labels.append(0 if i < n // 2 else 1)
    return np.array(sigs), np.array(labels), fs

### Starter 3：`# ============ 特征字典(每个特征: 函数+物理含义) ============`
（教师版：本段已是**完成态**，可直接演示讲解；学生版对应处为留白。）

In [ ]:
# ============ 特征字典(每个特征: 函数 + 物理含义) ============
# 教师版完成态: 这里保持空字典, 由下方 @feature 装饰器逐一登记
# 最终登记 30 个 = 时域10个(mean/std/max/ptp/rms/var/abs_mean/crest/kurtosis/clearance)
#              + 频域/统计20个(谱峰·带宽·能量分布13个 + 高阶矩·无量纲指标7个)
FEATURES = {}

### Starter 4：`def feature(name, meaning):`
（教师版：本段已是**完成态**，可直接演示讲解；学生版对应处为留白。）

In [ ]:
def feature(name, meaning):
    def deco(fn):
        FEATURES[name] = (fn, meaning)
        return fn
    return deco

### Starter 5：`# ---- 示例1-5(已给, 照此补齐其余) ----`
（教师版：本段已是**完成态**，可直接演示讲解；学生版对应处为留白。）

In [ ]:
# ---- 示例1-5(已给, 照此补齐其余) ----
@feature("mean", "均值: 振动能量平均水平")
def f_mean(x):
    return x.mean()

### Starter 6：`@feature("std", "标准差: 波动离散程度")`
（教师版：本段已是**完成态**，可直接演示讲解；学生版对应处为留白。）

In [ ]:
@feature("std", "标准差: 波动离散程度")
def f_std(x):
    return x.std()

### Starter 7：`@feature("max", "峰值: 最大振幅, 冲击敏感")`
（教师版：本段已是**完成态**，可直接演示讲解；学生版对应处为留白。）

In [ ]:
@feature("max", "峰值: 最大振幅, 冲击敏感")
def f_max(x):
    return np.abs(x).max()

### Starter 8：`@feature("ptp", "峰峰值: max-min, 磨损冲击幅值")`
（教师版：本段已是**完成态**，可直接演示讲解；学生版对应处为留白。）

In [ ]:
@feature("ptp", "峰峰值: max-min, 磨损冲击幅值")
def f_ptp(x):
    return np.ptp(x)

### Starter 9：`@feature("rms", "均方根: 有效值, 能量代表")`
（教师版：本段已是**完成态**，可直接演示讲解；学生版对应处为留白。）

In [ ]:
@feature("rms", "均方根: 有效值, 能量代表")
def f_rms(x):
    return np.sqrt((x ** 2).mean())

### Starter 10：`# ---- 时域补充5个（var / abs_mean / crest / 峭度 / 裕度）----`
（教师版：本段已是**完成态**，可直接演示；学生版对应处为留白。）

In [ ]:
# ---- 时域补充5个(第1个): 方差 ----
@feature("var", "方差: 偏离均值的二次矩, 等于std的平方")
def f_var(x):
    # 要求: 总体方差, 一次算完, 不要 for 循环累加
    # 教师版完成态
    return x.var()

### 💡 分层提示（先自己试5分钟再看）
**思路提示（按学生版留白顺序，一条对应一个空）**
> - **时域·方差 `f_var`**：总体方差 `x.var()`；想手写就 `((x - x.mean()) ** 2).mean()`，别用 for 累加。自检：应等于 `f_std` 的平方。
> - **时域·绝对均值 `f_abs_mean`**：先取绝对值再求均值 `np.abs(x).mean()`；和 `mean` 的差别是正负不抵消——信号带直流偏置时两者会几乎相等，这正是后面相关性去冗余要抓的线索。
> - **时域·峰值因子 `f_crest`**：`np.abs(x).max() / np.sqrt((x**2).mean())`；纯正弦应≈1.414，出现冲击会明显变大，拿它当自检标准。
> - **时域·峭度 `f_kurtosis`**：`((x-x.mean())**4).mean() / x.std()**4 - 3`（超额峭度，总体矩版本）。写成 `y=(x-x.mean())/(x.std()+1e-12)` 再 `(y*y*y*y).mean()-3` 更快；纯正弦≈-1.5，冲击越大越往正数走。特征字典里要写清是哪个版本，全班别混。
> - **时域·裕度 `f_clearance`**：`peak / (mean(sqrt(|x|)))**2`——顺序是"先开方→再取均值→最后平方"，写反就全错；它对早期点蚀比峭度更稳。
> - **`spectrum()` 单边幅值谱**：`amp = np.abs(np.fft.rfft(x)) * 2 / n`，`freqs = np.fft.rfftfreq(n, 1/fs)`。rfft 只给非负频率，除 0Hz 外都要 ×2 才补回共轭能量；DC 与（n 为偶数时）最后一个奈奎斯特谱线**不**翻倍。把口径写死在函数里，不要各人各写。
> - **主频 `f_dom_freq`**：`freqs[amp.argmax()]`，但**必须先去掉 DC**（`freqs[1:], amp[1:]`），否则带 0.5 直流偏置的信号永远把主频判成 0Hz——这是本题第一大坑。
> - **频谱能量 `f_spec_energy`**：去 DC 后 `(amp**2).sum()`；按帕塞瓦尔定理它应≈`2*x.var()`，可与时域方差互为校验，差得远就是谱归一化写错了。
> - **频域质心 `f_spec_centroid`**：把幅值归一化成权重 `w = amp/amp.sum()`，再 `(freqs*w).sum()`（等价 `np.average(freqs, weights=amp)`）；分母加 `1e-12` 防零。
> - **再补12个以上频域/统计特征**：套路都是"归一化权重 → 对频率加权"或"对幅值取矩"。频域可写：第2/第3大**局部谱峰**频率（先用 `(a[1:-1]>a[:-2])&(a[1:-1]>=a[2:])` 一次筛峰，再排序）、`spec_var`、`bandwidth=√spec_var`、均方根频率 `√(w·f²)`、90%能量频率（`np.cumsum` 后 `searchsorted`）、谱平坦度（几何均值/算术均值）、0-20Hz 与 20Hz 以上能量占比、主频±5Hz 带内能量。统计侧可写：偏度、变异系数、波形因子、谱熵、脉冲指标 `peak/mean(|x|)`、四分位裕度 `peak/(q75-q25)`（用 `np.partition` 比全排序快）、过零率。**注意**：像"半功率带宽"这类指标在单峰+噪声底的数据上会退化成恒定值（零方差列），要换成 90%能量频率。
> - **`extract()`**：`names = list(FEATURES)` 的登记顺序就是矩阵列顺序（保证可复现）；用 `fn.__code__.co_argcount > 1` 判断要不要传 `fs`；外层可以按信号循环（200 次），但单条信号内部一律向量化，别对 1000 个采样点写 Python 循环，否则 100ms 一定爆。
> - **`main()`·标准化+存盘**：`Z = (X - X.mean(0)) / (X.std(0) + 1e-12)`，再 `np.save('features.npy', Z)`、`np.save('labels.npy', labels)`；存盘前先检查有没有零方差列和 NaN/Inf。
> - **`main()`·箱线图**：`fig, axes = plt.subplots(2, 3)`，每个子图 `ax.boxplot([X[labels == 0, j], X[labels == 1, j]])` 取前 6 列；无显示器时先 `matplotlib.use("Agg")` 离屏渲染，或用 `np.percentile(X[...], [0,25,50,75,100])` 打五数概括表格代替。
> - **`main()`·相关性去冗余（A 层）**：`np.corrcoef(Z.T)` 得 30×30，取上三角找 `|r|>0.95` 的特征对，保留物理含义更明确的、剔除重复的，并在工单里写明"剔了谁、和谁重复、r 多少"；热力图上成片的亮方块就是冗余族。

| 层 | 这样做 |
|----|--------|
| **D 学徒** | 跟教师演示逐步照做：定位留白行 → 按上面思路逐句翻译成代码 → 运行看输出 → 对不上就看下方"报错自查" |
| **C 操作员** | 只看思路提示独立实现；卡住再对照教师课堂演示 |
| **B/A 组长·师傅** | 不看提示直接做；完成后做扩展：改参数/换实现对比效果，写进工单"参数-收益"留痕 |

**报错自查三板斧**：①读最后一行错误信息 → ②检查变量名/缩进/括号 → ③还不行问同层同学或教师
**自检**：对照上方“验收标准”四行逐条自查并在工单上签字，未做到的档位如实写「未做」——不得只勾 及格(D) 与 优秀(B+)：`及格(D)`＝`30特征计算正确；向量化为主；标准化正确；PEP 8≥75%` ｜ `良好(C)`＝`1000样本<100ms；特征字典含物理含义` ｜ `优秀(B)`＝`<50ms；广播运用充分；箱线图输出；文档完整` ｜ `超优(A)`＝`优秀+衍生特征5个+相关性特征选择(去冗余)`

### Starter 11：`@feature("abs_mean", "绝对均值")`
（教师版：本段已是**完成态**，可直接演示讲解；学生版对应处为留白。）

In [ ]:
# 教师版完成态: abs_mean —— 整流平均值, 与 mean 的差别是正负不抵消
@feature("abs_mean", "绝对均值: mean(|x|), 整流平均值, 对单向漂移敏感")
def f_abs_mean(x):
    return np.abs(x).mean()

### Starter 12：`@feature("crest", "脉冲指标: peak/rms")`
（教师版：本段已是**完成态**，可直接演示讲解；学生版对应处为留白。）

In [ ]:
@feature("crest", "脉冲指标(峰值因子): peak/rms, 冲击性核心指标")
def f_crest(x):
    # 要求: 峰值因子 = 绝对峰值 / 有效值; 纯正弦应≈1.414, 有冲击时明显变大
    # 教师版完成态
    return np.abs(x).max() / (np.sqrt((x ** 2).mean()) + 1e-12)

### Starter 13：`@feature("kurtosis", "峭度: 冲击性(四阶矩)")`
（教师版：本段已是**完成态**，可直接演示讲解；学生版对应处为留白。）

In [ ]:
@feature("kurtosis", "峭度: 四阶矩/std^4 - 3 (总体矩版本的超额峭度), 正弦≈-1.5")
def f_kurtosis(x):
    # 提示: ((x-x.mean())**4).mean() / std**4 - 3
    # 教师版完成态: 先标准化再连乘, 比 x**4 少一次幂运算(200条信号可省几毫秒)
    y = (x - x.mean()) / (x.std() + 1e-12)
    return float((y * y * y * y).mean() - 3.0)

### Starter 14：`@feature("clearance", "裕度指标: peak/半波平方均值^2")`
（教师版：本段已是**完成态**，可直接演示讲解；学生版对应处为留白。）

In [ ]:
# 教师版完成态: clearance —— 裕度指标, 分母是"先开方、再取均值、最后平方"
@feature("clearance", "裕度指标: peak/(mean(sqrt|x|))^2, 早期点蚀最灵敏")
def f_clearance(x):
    return np.abs(x).max() / (np.sqrt(np.abs(x)).mean() ** 2 + 1e-12)

### Starter 15：`# ---- 频域特征组（先做 spectrum 单边幅值谱工具）----`
（教师版：本段已是**完成态**，可直接演示；学生版对应处为留白。）

In [ ]:
# ---- 频域特征组: 先做谱工具(所有频域特征都复用它) ----
_SPECTRA = {}          # {id(信号): (信号引用, (freqs, amp))} —— 只做一次FFT的共享缓存


def spectrum(x, fs):
    """返回 freqs, amp: 单边幅值谱(已归一化)
    amp = |rfft(x)| * 2 / n —— rfft 只给非负频率, 除0Hz外都要 ×2 才补回共轭能量
    口径(写死在此, 避免全班口径混战): DC 不翻倍; n为偶数时最后的奈奎斯特 bin 也不翻倍
    教师版完成态: 加"同一条信号只做一次FFT"的缓存(按对象id), 13个频域特征共享谱线
    """
    hit = _SPECTRA.get(id(x))
    if hit is not None and hit[0] is x:          # 缓存命中: extract 里同一条信号反复调用
        return hit[1]
    n = x.size
    amp = np.abs(np.fft.rfft(x)) * 2.0 / n
    amp[0] *= 0.5                                # 0Hz(直流)不翻倍
    if n % 2 == 0:
        amp[-1] *= 0.5                           # 奈奎斯特频率不翻倍
    sp = (np.fft.rfftfreq(n, 1.0 / fs), amp)
    if len(_SPECTRA) > 1024:                     # 防止长跑内存膨胀
        _SPECTRA.clear()
    _SPECTRA[id(x)] = (x, sp)                    # 留住 x 的引用, id 才不会被复用
    return sp


def band(x, fs):
    """频域特征统一口径: 去掉 DC 后的 (freqs, amp) —— 见"提交要求与避坑"
    教师版完成态: 若不去DC, 带0.5直流偏置的信号永远把主频判成 0Hz
    """
    freqs, amp = spectrum(x, fs)
    return freqs[1:], amp[1:]


def w_freq(x, fs):
    """把幅值谱归一化成"频率的权重分布"(和为1), 质心/方差/熵都基于它
    教师版完成态
    """
    freqs, amp = band(x, fs)
    return freqs, amp / (amp.sum() + 1e-12)


def kth_peak(x, fs, k):
    """第k大"局部谱峰"的频率: 峰位用相邻比较一次性向量化筛出, 再按幅值排序取第k个
    教师版完成态: 直接取第k大 bin 会被泄漏/噪声底带偏, 所以先筛峰
    """
    freqs, amp = band(x, fs)
    pk = np.flatnonzero((amp[1:-1] > amp[:-2]) & (amp[1:-1] >= amp[2:])) + 1
    order = pk[np.argsort(amp[pk])[::-1]] if pk.size >= k else np.argsort(amp)[::-1]
    return float(freqs[order[k - 1]])

### Starter 16：`@feature("dom_freq", "主频率: 最大幅值对应频率")`
（教师版：本段已是**完成态**，可直接演示讲解；学生版对应处为留白。）

In [ ]:
@feature("dom_freq", "主频率: 去DC后最大幅值对应的频率(Hz)")
def f_dom_freq(x, fs=100):
    freqs, amp = band(x, fs)
    # 要求: 返回最大谱线所在的频率, 而不是最大幅值本身
    # 教师版完成态
    return float(freqs[int(amp.argmax())])

### Starter 17：`@feature("spec_energy", "频谱总能量")`
（教师版：本段已是**完成态**，可直接演示讲解；学生版对应处为留白。）

In [ ]:
@feature("spec_energy", "频谱总能量: 去DC后幅值平方和(帕塞瓦尔: =2×var, 与方差互校)")
def f_spec_energy(x, fs=100):
    # 要求: 用幅值谱求总能量, 不要重新做时域循环
    # 教师版完成态
    _, amp = band(x, fs)
    return float((amp ** 2).sum())

### Starter 18：`@feature("spec_centroid", "频域质心")`
（教师版：本段已是**完成态**，可直接演示讲解；学生版对应处为留白。）

In [ ]:
@feature("spec_centroid", "频域质心: 幅值加权平均频率(Hz), 能量中心位置")
def f_centroid(x, fs=100):
    # 要求: sum(f*amp)/sum(amp)
    # 教师版完成态
    freqs, w = w_freq(x, fs)
    return float((freqs * w).sum())

### Starter 19：`# ... 再补12个以上频域/统计特征`
（教师版：骨架注释里点名的 f2/f3、bandwidth、spec_var、谱熵、偏度、变异系数、波形因子、脉冲指标、四分位裕度、过零率、90%能量频率、主频带能量……已**全部实现**，共补了17个，使特征总数达到30维。学生版此格只有一段清单注释。）

In [ ]:
# ... 再补12个以上频域/统计特征 —— 教师版完成态(本组: 谱峰与带宽族, 7个)
# 骨架注释点名的 f2/f3(第2、3峰频率)、bandwidth、spec_var、均方根频率、谱平坦度都在此,
# 另加 f90_energy: 半功率带宽在本数据上恒为1条谱线宽(零方差, 模型里没信息),
# 故改用"累计功率首次达到90%的频率", 既承接同一个物理问题又真正可用 —— 这是A层要讨论的坑。
@feature("f2", "第2谱峰频率: 第2大局部峰(Hz), 故障特征频率常在这里找")
def f_f2(x, fs=100):
    return kth_peak(x, fs, 2)


@feature("f3", "第3谱峰频率: 第3大局部峰(Hz), 与f2联合看谐波/边带")
def f_f3(x, fs=100):
    return kth_peak(x, fs, 3)


@feature("spec_var", "频域方差: 幅值加权的频率方差(Hz^2), 能量分散程度")
def f_spec_var(x, fs=100):
    freqs, w = w_freq(x, fs)
    return float((w * (freqs - (freqs * w).sum()) ** 2).sum())


@feature("bandwidth", "带宽: 频域标准差 sqrt(spec_var) (Hz)")
def f_bandwidth(x, fs=100):
    freqs, w = w_freq(x, fs)
    return float(np.sqrt((w * (freqs - (freqs * w).sum()) ** 2).sum()))


@feature("spec_rms_freq", "均方根频率: sqrt(加权f^2) (Hz), 高频成分整体水平")
def f_spec_rms_freq(x, fs=100):
    freqs, w = w_freq(x, fs)
    return float(np.sqrt((w * freqs ** 2).sum()))


@feature("f90_energy", "90%能量频率: 累计功率首次达到90%的频率(Hz)")
def f_f90(x, fs=100):
    freqs, amp = band(x, fs)
    cum = np.cumsum(amp ** 2) / ((amp ** 2).sum() + 1e-12)
    return float(freqs[int(np.searchsorted(cum, 0.90))])


@feature("spec_flatness", "谱平坦度: 功率的几何均值/算术均值(0~1), 越接近1越像噪声")
def f_spec_flatness(x, fs=100):
    _, amp = band(x, fs)
    p = amp ** 2 + 1e-12
    return float(np.exp(np.log(p).mean()) / p.mean())

### Starter 19b（教师版新增·完成态）：补充组② —— 能量分布族（3个）

In [ ]:
# 补充组②(教师版完成态): 能量分布族, 3个 —— 工况越差能量越往高频搬
@feature("energy_ratio_low", "低频能量占比: 0-20Hz功率/总功率, 基频主导程度")
def f_energy_low(x, fs=100):
    freqs, amp = band(x, fs)
    return float((amp[freqs <= 20] ** 2).sum() / ((amp ** 2).sum() + 1e-12))


@feature("energy_ratio_high", "高频能量占比: 20Hz~奈奎斯特功率/总功率, 磨损敏感")
def f_energy_high(x, fs=100):
    freqs, amp = band(x, fs)
    return float((amp[freqs > 20] ** 2).sum() / ((amp ** 2).sum() + 1e-12))


@feature("dom_band_energy", "主频带能量: 主频±5Hz带内的功率和(绝对量, 与占比互补)")
def f_dom_band_energy(x, fs=100):
    freqs, amp = band(x, fs)
    f0 = freqs[int(amp.argmax())]
    return float((amp[np.abs(freqs - f0) <= 5] ** 2).sum())

### Starter 19c（教师版新增·完成态）：补充组③ —— 高阶矩与无量纲指标（7个）

In [ ]:
# 补充组③(教师版完成态): 高阶矩与无量纲指标, 7个 —— 不随幅值/量程变化, 跨设备可比
@feature("skewness", "偏度: 三阶矩/std^3 (总体矩版本), 单向磨损造成的分布偏移")
def f_skewness(x):
    y = (x - x.mean()) / (x.std() + 1e-12)
    return float((y * y * y).mean())


@feature("cv", "变异系数: std/|mean|, 无量纲离散度, 便于跨工况比较")
def f_cv(x):
    return float(x.std() / (np.abs(x.mean()) + 1e-12))


@feature("shape_factor", "波形因子: rms/mean(|x|), 纯正弦≈1.111, 与crest联合看冲击形状")
def f_shape_factor(x):
    return float(np.sqrt((x ** 2).mean()) / (np.abs(x).mean() + 1e-12))


@feature("spec_entropy", "谱熵: 归一化功率的Shannon熵, 越小能量越集中在少数谱线")
def f_spec_entropy(x, fs=100):
    _, amp = band(x, fs)
    p = amp ** 2 / ((amp ** 2).sum() + 1e-12)
    return float(-(p * np.log(p + 1e-12)).sum())


@feature("impulse", "脉冲指标: peak/mean(|x|), 冲击类故障放大系数")
def f_impulse(x):
    return float(np.abs(x).max() / (np.abs(x).mean() + 1e-12))


@feature("margin", "四分位裕度: peak/(q75-q25), 抗噪稳健的离群冲击指标")
def f_margin(x):
    n = x.size
    lo = np.partition(x, n // 4)[n // 4]          # O(n)选择替代全排序: 200条信号省下几毫秒
    hi = np.partition(x, 3 * n // 4)[3 * n // 4]
    return float(np.abs(x).max() / (hi - lo + 1e-12))


@feature("zcr", "过零率: 去均值后符号变化的比例, 频率成分的粗略代理")
def f_zcr(x):
    return float((np.diff(np.sign(x - x.mean())) != 0).mean())

### Starter 19d（教师版新增·完成态）：main() 依赖的报告助手

In [ ]:
# ---- 教师版完成态: main() 需要的报告助手(全离线, 只算不写盘) ----
import io


def zscore(X):
    """Z-score标准化: 每列减均值除标准差; 标准差为0的列保持0(不产生NaN)"""
    mu, sd = X.mean(0), X.std(0)
    return (X - mu) / np.where(sd > 0, sd, 1.0)


def npy_selfcheck(Z, labels):
    """对应"保存 features.npy + labels.npy": 在内存里完成 np.save/np.load 往返校验
    教师版刻意不落盘 —— 课程包根目录已有示例 features.npy/labels.npy, 直接覆盖会破坏其它工单
    """
    buf = io.BytesIO()
    np.save(buf, Z)
    np.save(buf, labels)
    buf.seek(0)
    Z2, L2 = np.load(buf), np.load(buf)
    ok = Z2.shape == Z.shape and np.allclose(Z2, Z) and np.array_equal(L2, labels)
    return buf.tell(), ok


def class_separation(X, names, labels, top=6):
    """类间可分性: Cohen's d = |μ磨损-μ正常| / 合并标准差, 越大越能分开两类"""
    rows = []
    for j, nm in enumerate(names):
        a, b = X[labels == 0, j], X[labels == 1, j]
        pooled = np.sqrt((a.var() + b.var()) / 2.0)
        d = float('inf') if pooled < 1e-9 else float(abs(b.mean() - a.mean()) / pooled)
        rows.append((d, nm, float(a.mean()), float(b.mean())))
    rows.sort(key=lambda r: -r[0])
    return rows[:top], rows


def five_number(X, names, labels, cols):
    """箱线图的五数概括(与图同源, 无图形环境时用表格呈现同一份数据)"""
    out = []
    for j in cols:
        for lab, tag in ((0, '正常'), (1, '磨损')):
            q = np.percentile(X[labels == lab, j], [0, 25, 50, 75, 100])
            out.append((names[j], tag, *[float(v) for v in q]))
    return out


def redundancy(Z, names, thr=0.95):
    """A层拓展: 相关系数矩阵去冗余 —— |r|>thr 时保留先登记的, 剔除后出现的
    返回 (保留名单, 被剔除[(特征, 与谁重复, r)], 零方差列, |r|最大的前5对)
    """
    live = [j for j in range(len(names)) if Z[:, j].std() > 1e-12]
    dead = [names[j] for j in range(len(names)) if j not in live]
    C = np.nan_to_num(np.corrcoef(Z[:, live].T))
    kept, dropped = [], []
    for jj, j in enumerate(live):
        hit = [i for i, k in enumerate(kept) if abs(C[jj, k]) > thr]
        if hit:
            dropped.append((names[j], names[live[hit[0]]], float(C[jj, hit[0]])))
        else:
            kept.append(jj)
    pairs = sorted(([abs(C[i, j]), names[live[i]], names[live[j]], float(C[i, j])]
                    for i in range(len(live)) for j in range(i + 1, len(live))), key=lambda r: -r[0])
    return [names[live[k]] for k in kept], dropped, dead, pairs[:5]

### Starter 19e（教师版新增·完成态）：箱线图与相关系数热力图（无显示器环境自动降级）

In [ ]:
# ---- 教师版完成态: 可视化(箱线图 + 相关系数热力图), 无显示环境也能跑 ----
try:
    _IP = get_ipython()                 # 只有真实 notebook 内核里才有这个函数
except NameError:
    _IP = None                          # 纯 python / 判题脚本执行

try:
    import matplotlib
    import matplotlib.font_manager as fm
    if _IP is None:
        matplotlib.use("Agg")           # 离屏渲染: 不弹窗、不需要显示器
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:                     # 机器没装 matplotlib 时的降级: 只输出五数概括表格
    plt, HAS_MPL = None, False


def _use_cjk_font():
    """注册课程包自带中文字体 fonts/ZH_Font.ttf(cell2 已把 cwd 切到课程包根目录);
    找不到就退回系统常见中文字体, 再没有就返回 False —— 图内改用英文标签, 免得满屏 Glyph missing"""
    if not HAS_MPL:
        return False
    if os.path.exists("fonts/ZH_Font.ttf"):
        try:
            fm.fontManager.addfont("fonts/ZH_Font.ttf")
            matplotlib.rcParams["font.sans-serif"] = [
                fm.FontProperties(fname="fonts/ZH_Font.ttf").get_name(), "DejaVu Sans"]
            matplotlib.rcParams["axes.unicode_minus"] = False
            return True
        except Exception as e:                     # 字体坏损/权限问题也不能让绘图挂掉
            print("  [字体] ZH_Font.ttf 注册失败(%s), 转用系统字体" % e)
    names = {f.name for f in fm.fontManager.ttflist}
    for cand in ("PingFang SC", "Microsoft YaHei", "Noto Sans CJK SC", "SimHei", "Arial Unicode MS"):
        if cand in names:
            matplotlib.rcParams["font.sans-serif"] = [cand, "DejaVu Sans"]
            matplotlib.rcParams["axes.unicode_minus"] = False
            return True
    return False


CN_OK = _use_cjk_font()


def zh(cn, en):
    """有中文字体给中文标签, 否则给英文标签(同一张图, 两种口径)"""
    return cn if CN_OK else en


def show_fig(fig, title):
    """notebook 内核里内联显示; 离线执行时保存为PNG字节流自检, 数据改由表格打印"""
    if fig is None:
        print(f"  [降级] 无 matplotlib, {title} 改用下方五数概括表格")
        return
    if _IP is not None:
        from IPython.display import display
        display(fig)
        plt.close(fig)
        return
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=80)
    plt.close(fig)
    print(f"  [离屏] {title}: PNG {buf.tell():,} 字节(同一份数据见下方表格), 中文字体={CN_OK}")


def plot_box6(X, names, labels, cols):
    """每类对比前6个特征的箱线图(2×3)"""
    if not HAS_MPL:
        return None
    fig, axes = plt.subplots(2, 3, figsize=(11, 6), squeeze=False)
    for ax, j in zip(axes.ravel(), cols):
        ax.boxplot([X[labels == 0, j], X[labels == 1, j]], patch_artist=True)
        ax.set_xticks([1, 2])
        ax.set_xticklabels([zh("正常", "Normal"), zh("磨损", "Worn")])
        ax.set_title(zh(f"{names[j]}: {FEATURES[names[j]][1].split(':')[0]}", names[j]), fontsize=9)
        ax.grid(alpha=.3)
    fig.suptitle(zh(f"P2.2 前6维特征箱线图 (正常 n={(labels == 0).sum()} vs 磨损 n={(labels == 1).sum()})",
                    f"P2.2 first-6 features boxplot (Normal n={(labels == 0).sum()} vs Worn n={(labels == 1).sum()})"),
                 fontsize=11)
    fig.tight_layout()
    return fig


def plot_corr_heatmap(Z, names):
    """30×30 相关系数热力图: 成片的亮块就是冗余特征族(A层依据)"""
    if not HAS_MPL:
        return None
    C = np.nan_to_num(np.corrcoef(Z.T))
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(C, cmap="coolwarm", vmin=-1, vmax=1)
    ax.set_xticks(range(len(names)))
    ax.set_yticks(range(len(names)))
    ax.set_xticklabels(names, rotation=90, fontsize=6)
    ax.set_yticklabels(names, fontsize=6)
    ax.set_title(zh("特征相关系数矩阵 (|r|>0.95 判冗余)", "Feature correlation matrix (|r|>0.95 = redundant)"),
                 fontsize=10)
    fig.colorbar(im, ax=ax, shrink=.8, label=zh("相关系数 r", "corr r"))
    fig.tight_layout()
    return fig

### Starter 20：`def extract(signals, fs=100):`
（教师版：本段已是**完成态**，可直接演示讲解；学生版对应处为留白。）

In [ ]:
def extract(signals, fs=100):
    """对每条信号计算全部特征 → 特征矩阵(N×30) + 名称列表
    要求: 列顺序 = FEATURES 登记顺序(可复现); 计时只包特征计算
    教师版完成态: 外层循环是"每条信号"(200次), 单条信号内部一律向量化
    """
    names = list(FEATURES)
    fns = [(FEATURES[n][0], FEATURES[n][0].__code__.co_argcount > 1) for n in names]
    t0 = time.time()
    X = np.empty((len(signals), len(names)), dtype=float)
    for k, s in enumerate(signals):
        for j, (fn, need_fs) in enumerate(fns):
            X[k, j] = fn(s, fs) if need_fs else fn(s)
    ms = (time.time() - t0) * 1000
    _SPECTRA.clear()                      # 释放谱缓存(缓存按信号id, 必须在算完后清)
    return X, names, ms

### Starter 21：`def main():`
（教师版：本段已是**完成态**，可直接演示讲解；学生版对应处为留白。）

In [ ]:
def main():
    sigs, labels, fs = make_signals()
    X, names, ms = extract(sigs, fs)
    verdict = "优秀(<50ms)" if ms < 50 else ("及格(<100ms)" if ms < 100 else "未达标: 检查是否退化成逐点for循环")
    print(f"特征矩阵: {X.shape}  特征数: {len(names)}  计算耗时: {ms:.0f}ms → {verdict}")
    print(f"特征字典物理含义齐全: {all(FEATURES[n][1] for n in names)}")
    assert np.isfinite(X).all(), "出现NaN/Inf: 检查除零保护(1e-12)是否漏写"
    assert X.shape == (len(sigs), 30), "特征数应为30"
    # 教师版完成态-1: Z-score标准化 → features.npy + labels.npy(内存内往返校验, 不覆盖课程包示例文件)
    Z = zscore(X)
    nbytes, ok = npy_selfcheck(Z, labels)
    dead = [n for n, j in zip(names, range(len(names))) if X[:, j].std() < 1e-12]
    print(f"① 标准化: 列均值最大|·|={np.abs(Z.mean(0)).max():.1e} 列标准差均值={Z.std(0).mean():.3f} "
          f"| 零方差特征={dead or '无'} | npy序列化 {nbytes:,}B 往返一致={ok}")
    print("   生产写法: np.save('features.npy', Z); np.save('labels.npy', labels)")
    # 教师版完成态-2: 箱线图(matplotlib, 每类对比前6个特征)
    cols = list(range(6))
    show_fig(plot_box6(X, names, labels, cols), "前6维特征箱线图")
    print("② 前6个特征五数概括 (min/Q1/中位/Q3/max):")
    for nm, tag, v0, q1, q2, q3, v1 in five_number(X, names, labels, cols):
        print(f"   {nm:<10}{tag}  min={v0:8.4f}  Q1={q1:8.4f}  中位={q2:8.4f}  Q3={q3:8.4f}  max={v1:8.4f}")
    # 教师版完成态-3: 类间可分性(哪些特征真能把正常/磨损分开)
    top, all_rows = class_separation(X, names, labels, top=6)
    print("③ 类间可分性 Top6 (Cohen's d = |μ磨损-μ正常|/合并σ):")
    for d, nm, m0, m1 in top:
        ds = "∞ 类内恒定→单特征100%可分(注意过拟合)" if np.isinf(d) else f"{d:6.2f}"
        print(f"   {nm:<18} d={ds:<28} 正常={m0:9.4f} 磨损={m1:9.4f}")
    # 教师版完成态-4: (超优)相关系数矩阵热力图 + 剔除|r|>0.95的冗余特征
    show_fig(plot_corr_heatmap(Z, names), "30维特征相关系数热力图")
    kept, dropped, zero_var, pairs = redundancy(Z, names, thr=0.95)
    print("④ 相关性 |r| 最大5对: " + ", ".join(f"{a}~{b}={r:+.3f}" for _, a, b, r in pairs))
    print(f"   剔除|r|>0.95后保留 {len(kept)}/{len(names)}: {kept}")
    print(f"   剔除 {len(dropped)} 个冗余特征(前6个): " + ", ".join(f"{n}~{p}(|r|={abs(r):.3f})"
                                                          for n, p, r in dropped[:6]))
    return X, labels

### Starter 22：`if __name__ == "__main__":`
（教师版：本段已是**完成态**，可直接演示讲解；学生版对应处为留白。）

In [ ]:
if __name__ == "__main__":
    main()

## C层·达标证据：30 特征正确 + 向量化为主（工单 C 层达标项）

工单 C 层写的是 **「30特征正确 + 向量化为主」**。"正确"两个字不能由特征库自己说了算——
同一个 bug 在分子分母里重复出现时，自检是察觉不到的。所以本 cell 换一套**完全不相干的算法**
（纯 Python 三趟 `for` 累加 + `sorted()` 全排序）把 15 个时域特征重算一遍，再叠四层旁证：

| 证据 | 校的是什么 | 口径 |
|---|---|---|
| ① 独立算法交叉校验 | 15 个时域特征的**数值** | 3 条信号 × 15 = 45 格，最大相对误差 < 1e-6 |
| ② 内部恒等式 | 特征之间的**定义关系** | `var=std²`、`rms²=mean²+var`、`crest=peak/rms`、`impulse`、`波形因子` |
| ③ 理论锚点 | 骨架注释里承诺的**物理常识** | A=3、8Hz 纯正弦：crest≈√2、波形因子≈π/(2√2)（极值类只到 1% 容差，因为每周期仅 12.5 个采样点）、峭度=−1.5（严格） |
| ④ 时域↔频域互校 | 谱归一化口径（帕塞瓦尔） | 去 DC 后谱总能量 = 2×时域方差 |
| ⑤ 向量化 | "向量化为主"这句 | 15 个时域函数字节码里 `FOR_ITER` 计数 = 0 |

> `margin` 那条顺带校了 `np.partition` 的 O(n) 选择捷径：特征库取分区后的位次，
> 参考实现取全排序后的同一位次，两者必须一致，否则就是索引写偏了。
> 时域 15 + 频域 15 = 30 的分组口径与 `工程化项目/p22_feature_factory`、`教师参考/solutions.py::p22` 一致。


In [ ]:
# ---- C层·达标证据: 工单写"30特征正确 + 向量化为主", 本 cell 把这两句话变成可断言的数字 ----
# 正确性不能靠"特征库自己算自己"核对, 所以这里另写一套参考实现: 纯 Python 三趟 for 循环
# + sorted() 全排序 —— 与特征库的 numpy 一行式(以及 np.partition 的 O(n) 选择)零共享。
import math
import dis

TIME15 = ["mean", "std", "max", "ptp", "rms", "var", "abs_mean", "crest",
          "kurtosis", "clearance", "skewness", "cv", "shape_factor", "impulse", "margin"]


def ref_time15(x):
    """15个时域特征的教科书定义版(逐点累加, 不用 numpy)。"""
    n = len(x)
    s = sa = s2 = amax = srt = 0.0
    hi, lo = -math.inf, math.inf
    for v in x:
        a = abs(v)
        s += v
        sa += a
        s2 += v * v
        hi = max(hi, v)
        lo = min(lo, v)
        amax = max(amax, a)
    m = s / n
    m2 = m3 = m4 = 0.0
    for v in x:
        d = v - m
        m2 += d * d
        m3 += d * d * d
        m4 += d * d * d * d
        srt += math.sqrt(abs(v))
    m2 /= n
    m3 /= n
    m4 /= n
    std, rms, abs_mean = math.sqrt(m2), math.sqrt(s2 / n), sa / n
    b = sorted(x)                      # 特征库用 np.partition, 这里全排序取同一位次, 校的是那条捷径
    eps = 1e-12
    return {"mean": m, "std": std, "max": amax, "ptp": hi - lo, "rms": rms,
            "var": m2, "abs_mean": abs_mean, "crest": amax / (rms + eps),
            "kurtosis": m4 / (m2 * m2 + eps) - 3.0,
            "clearance": amax / ((srt / n) ** 2 + eps),
            "skewness": m3 / (std ** 3 + eps), "cv": std / (abs(m) + eps),
            "shape_factor": rms / (abs_mean + eps), "impulse": amax / (abs_mean + eps),
            "margin": amax / (b[3 * n // 4] - b[n // 4] + eps)}


sigs, labels, fs = make_signals()
X, names, ms = extract(sigs, fs)
freq15 = [n for n in names if n not in TIME15]
assert len(TIME15) == len(freq15) == 15 and len(names) == 30, (len(TIME15), len(freq15), len(names))
Xt = X[:, [names.index(n) for n in TIME15]]
print(f"结构: 时域15 + 频域15 = 30 个特征 ｜ 时域子矩阵 shape={Xt.shape}（从30列按名取15列） 全矩阵 shape={X.shape}")

probe = (0, 99, 199)
worst = 0.0
for k in probe:
    ref = ref_time15(sigs[k])
    for nm in TIME15:
        worst = max(worst, abs(X[k, names.index(nm)] - ref[nm]) / (abs(ref[nm]) + 1e-12))
print(f"① 独立算法交叉校验 {len(probe)}条信号 × 15特征 = {len(probe) * 15} 格: 最大相对误差 {worst:.1e}")
assert worst < 1e-6, worst

ref0 = ref_time15(sigs[0])
print("   第1条信号逐条对照(特征库 / 参考 / 相对误差):")
for nm in TIME15:
    v = X[0, names.index(nm)]
    print(f"   {nm:<12} {v:>11.5f} {ref0[nm]:>11.5f} {abs(v - ref0[nm]) / (abs(ref0[nm]) + 1e-12):>9.1e}")


def col(nm):
    return X[:, names.index(nm)]


e1 = np.abs(col("var") - col("std") ** 2).max() / col("var").max()
e2 = np.abs(col("rms") ** 2 - (col("mean") ** 2 + col("var"))).max() / (col("rms") ** 2).max()
e3 = np.abs(col("crest") - col("max") / (col("rms") + 1e-12)).max() / col("crest").max()
e4 = np.abs(col("impulse") - col("max") / (col("abs_mean") + 1e-12)).max() / col("impulse").max()
e5 = np.abs(col("shape_factor") - col("rms") / (col("abs_mean") + 1e-12)).max() / col("shape_factor").max()
print(f"② 内部恒等式(200条×5组, 相对误差): var=std² {e1:.1e} | rms²=mean²+var {e2:.1e} | "
      f"crest=peak/rms {e3:.1e} | impulse=peak/abs_mean {e4:.1e} | 波形因子=rms/abs_mean {e5:.1e}")
assert max(e1, e2, e3, e4, e5) < 1e-10

# 锚点与教师参考附录 p22() 同一条正弦(A=3, 8Hz, 窗内80个整周期): 每周期只有12.5个采样点,
# 采样点不会严格落在波峰上, 所以"极值类"指标只能收到 1% 容差——这是口径, 不是实现误差。
sine = 3.0 * np.sin(2 * np.pi * 8 * np.arange(1000) / 100.0)
anchors = {nm: float(FEATURES[nm][0](sine)) for nm in ("crest", "shape_factor", "kurtosis")}
print(f"③ 理论锚点(A=3、8Hz纯正弦): crest={anchors['crest']:.4f}(√2=1.4142) "
      f"波形因子={anchors['shape_factor']:.4f}(π/2√2=1.1107) 峭度={anchors['kurtosis']:.4f}(应=-1.5000)")
assert abs(anchors["crest"] - math.sqrt(2)) < 1e-2, anchors
assert abs(anchors["shape_factor"] - math.pi / (2 * math.sqrt(2))) < 1e-2, anchors
assert abs(anchors["kurtosis"] + 1.5) < 1e-9, anchors

# 帕塞瓦尔: 单边谱去掉 DC 后, 谱总能量 = 2×时域方差; 唯一残差是奈奎斯特 bin 不翻倍那一项。
pe = np.abs(col("spec_energy") - 2 * col("var")) / (2 * col("var"))
print(f"④ 时域↔频域互校(帕塞瓦尔: 谱总能量=2×时域方差, 残差来自奈奎斯特bin不翻倍): "
      f"200条最大相对误差 {pe.max():.1e}")
assert pe.max() < 5e-3, pe.max()

loops = [nm for nm in TIME15 if any(op.opname == "FOR_ITER"
                                    for op in dis.get_instructions(FEATURES[nm][0]))]
print(f"⑤ 向量化: 15个时域特征函数中含 Python 逐点循环(FOR_ITER 字节码)的 = {len(loops)} 个 {loops or ''}")
assert not loops, loops
print(f"→ C层达标: 时域15维逐条对上另一套算法 + 5组恒等式 + 3个理论锚点 + 频域互校, "
      f"且单条信号内部无逐点 for(外层只按信号循环 {len(sigs)} 次) ｜ 本轮耗时 {ms:.0f}ms")


## A层·进阶拓展：性能优化对比（工单A层达标项）

工单 A 层要求 **特征提取 <100ms（优秀 <50ms）+ 特征字典完整 + 相关去冗余**。
这条只能用当堂实测证明，所以本 cell 做三件可比的事：
1. **同口径对比**：同样 10 个时域特征，纯 Python 逐点循环 vs numpy 向量化，报提速倍数；
2. **边际成本**：再加 20 个频域/统计特征只多花多少毫秒（单边谱只做一次 rfft 并缓存复用）；
3. **吞吐折算**：把 200 条实测折算成 µs/特征·条，再推 3000 行的量级，便于跨机器比较。

> 诚实口径：绝对毫秒数与机器负载强相关，因此同时打印首跑/最优/中位，
> 并要求课堂上当堂重跑取最优；未达 50ms 时如实写“未进50ms”，不编造达标数字。


In [ ]:
def bench(repeat=5):
    """A层性能实测: 同一批特征, 纯Python逐点 vs numpy向量化(同口径可比)
    再看20个频域特征的边际成本——谱缓存把30个特征的总账压到多少
    """
    import math
    sigs, labels, fs = make_signals()
    names = list(FEATURES)
    time_names = names[:10]                       # 前10个=纯时域, 可用同一公式两种写法算

    def naive(s):                                 # 基线: 逐点Python, 只算同样的10个时域特征
        x = [float(v) for v in s]
        n = len(x)
        mu = sum(x) / n
        var = sum((v - mu) ** 2 for v in x) / n
        sd = math.sqrt(var)
        peak, low = max(x), min(x)
        rms = math.sqrt(sum(v * v for v in x) / n)
        abs_mean = sum(abs(v) for v in x) / n
        m4 = sum((v - mu) ** 4 for v in x) / n
        sqrt_abs_mean = sum(math.sqrt(abs(v)) for v in x) / n
        return {"mean": mu, "std": sd, "max": peak, "ptp": peak - low, "rms": rms,
                "var": var, "abs_mean": abs_mean, "crest": peak / (rms + 1e-12),
                "kurtosis": m4 / (sd ** 4 + 1e-12) - 3.0,
                "clearance": peak / (sqrt_abs_mean ** 2 + 1e-12)}

    t0 = time.time()
    for s in sigs:
        naive(s)
    naive_ms = (time.time() - t0) * 1000

    fns = [(FEATURES[nm][0], FEATURES[nm][0].__code__.co_argcount > 1) for nm in time_names]
    t0 = time.time()
    for _ in range(repeat):
        for s in sigs:
            for fn, need_fs in fns:
                fn(s, fs) if need_fs else fn(s)
    vec10_ms = (time.time() - t0) * 1000 / repeat

    runs = [extract(sigs, fs)[2] for _ in range(repeat)]
    best, med = min(runs), sorted(runs)[repeat // 2]
    per_signal = best / len(sigs)

    print("同口径10个时域特征 x %d条: 纯Python逐点 %.0fms | numpy向量化 %.0fms | 提速 %.1fx"
          % (len(sigs), naive_ms, vec10_ms, naive_ms / vec10_ms))
    print("全部30特征 x %d条(重复%d次): 首跑%.0fms 最优%.0fms 中位%.0fms → 单条%.3fms"
          % (len(sigs), repeat, runs[0], best, med, per_signal))
    print("边际成本: 后20个频域/统计特征共 %.0fms(=%.0f%%), 因为一条信号只做1次rfft并缓存复用"
          % (best - vec10_ms, 100 * (best - vec10_ms) / best))
    print("吞吐口径: %.1fµs/特征·条 → 折算3000行(=3000条)x30特征约 %.0fms"
          % (per_signal * 1000 / 30, per_signal * 3000))
    gate = ("达标(<50ms)" if best < 50 else
            "本机未进50ms, 仍<100ms及格" if best < 100 else
            "本机超100ms: 检查是否退化成逐点for循环/重复FFT")
    print("A层闸门(工单口径3000行<100ms, 优秀<50ms): 本次200条实测 %.0fms → %s" % (best, gate))
    print("诚实提示: 绝对毫秒与机器负载相关, 本cell打印首跑/最优/中位三个值, 课堂当堂重跑取最优")
    assert naive_ms > vec10_ms, "向量化必须快于同口径纯Python基线"
    assert len(FEATURES) == 30 and all(FEATURES[nm][1] for nm in FEATURES), "A层要求30项特征字典齐全"
    print("特征字典: 30 项, 物理含义全部登记 ✓")
    return {"naive": naive_ms, "vec10": vec10_ms, "runs": runs}


BENCH = bench()


## 四、提交要求与避坑
- 提交: `学号_姓名_P2.2.py` 或本notebook(含输出)
- ⚠ rfft已含共轭对称，别再取一半
- ⚠ 幅值×2/n补偿能量，物理量纲才对
- ⚠ 峭度/偏度公式版本写进特征字典，避免口径混战

## 教师参考答案（来源：`教师参考/solutions.py` · `p22()`）

> **发放纪律：本附录勿随学生版下发。** 课堂按层放提示（D层演示照做/C层只给思路行/A层直接做）。

学生版每个留白处 → 教师版完成态 cell → 下方 `p22()` 参考实现，逐条对应（17 个留白全覆盖）：

| 学生版留白 | 学生版 cell | 教师版 cell | 完成态要点 | `p22()` 对应写法 |
|---|---|---|---|---|
| `f_var()` 时域·方差 | 23 | 23 | 总体方差 `x.var()`，自检 = `std**2` | `F["var"]` |
| `f_abs_mean()` 时域·绝对均值 | 26 | 26 | `np.abs(x).mean()` 整流平均；与 mean 高度相关（去冗余案例） | `F["abs_mean"]` |
| `f_crest()` 时域·峰值因子 | 28 | 28 | peak/rms + 1e-12 防零，纯正弦≈1.414 | `F["crest"]` |
| `f_kurtosis()` 时域·峭度 | 30 | 30 | 先标准化再连乘 `(y*y*y*y).mean()-3`，正弦≈-1.5 | `F["kurtosis"]`+`mom()` |
| `f_clearance()` 时域·裕度 | 32 | 32 | peak/(mean(√｜x｜))²，开方-取均值-平方顺序不能错(｜为全角, 不撑破表格) | `F["clearance"]` |
| `spectrum()` 单边幅值谱 | 34 | 34 | ×2/N 补偿；DC 与奈奎斯特 bin 不翻倍；同信号共享一次 FFT；附 `band/w_freq/kth_peak` | `spectrum()/band()/wf()/kp()` |
| `f_dom_freq()` 主频率 | 36 | 36 | 去 DC 后 `freqs[amp.argmax()]`，否则永远判成 0Hz | `F["dom_freq"]` |
| `f_spec_energy()` 频谱总能量 | 38 | 38 | 去 DC 后 `(amp**2).sum()` ≈ 2×var（帕塞瓦尔互校） | `F["spec_energy"]` |
| `f_centroid()` 频域质心 | 40 | 40 | 幅值归一化成权重后 `(freqs*w).sum()` | `F["spec_centroid"]` |
| 再补12个以上·谱峰与带宽族(7个) | 42 | 42 | f2/f3 取局部谱峰；spec_var/bandwidth/均方根频率；f90_energy 顶替退化的半功率带宽；谱平坦度 | `F["f2"…"spec_flatness"]` |
| 再补12个以上·能量分布族(3个) | 42 | 44 | 0-20Hz / 20Hz以上 / 主频±5Hz 带内能量，磨损时高频占比上移 | `F["energy_ratio_*"]` |
| 再补12个以上·统计族(7个) | 42 | 46 | 偏度/变异系数/波形因子/谱熵/脉冲指标/四分位裕度(`np.partition`)/过零率 | `F["skewness"…"zcr"]` |
| `extract()` 特征矩阵 | 44 | 52 | 登记顺序=列顺序；按 `co_argcount` 决定是否传 fs；预分配 `np.empty` | 断言C 的 X 构造 |
| `main()` Z-score + 存盘 | 46 | 48 | `zscore()` + `npy_selfcheck()`（内存内 np.save/np.load 往返，不覆盖课程包示例文件） | 断言C 的 Z |
| `main()` 箱线图 | 46 | 50 | `plot_box6/five_number/show_fig`：有内核就内联，没显示器就 Agg 离屏+表格 | 断言A/B 的五数对比 |
| `main()` 相关性去冗余(A层) | 46 | 48 | `redundancy()` 剔除冗余对(｜r｜>0.95) + `plot_corr_heatmap()`；零方差列单独报告 | 断言C 的 r(低,高)=-1 |
| `main()` 汇总打印 | 46 | 54 | 计时判定 + 类间可分性 Top6(Cohen d) + 冗余清单 | `class_separation` 等价 d |

把上面 17 处全部填完，`main()` 就能离线跑出 `(200, 30)` 的特征矩阵、类间可分性与冗余清单——即"及格=30特征正确+向量化为主"，耗时判定与"优秀<50ms"由打印行给出。

### 下方参考实现覆盖的检查点
- 纯正弦物理真值：rms=3/√2、var=4.5、crest≈√2、峭度=-1.5、偏度=0、波形因子≈π/(2√2)、主频=8Hz、spec_energy=2×var；
- 磨损型信号（33Hz冲击串+60Hz）后：峭度/峰值因子/裕度/脉冲指标/四分位裕度/过零率/谱能量/谱熵/高频占比/带宽 全部同向变大；
- 50×30 流水线：无 NaN/Inf、无零方差列、Z-score 后列均值=0 列标准差=1、`r(低频占比,高频占比)=-1`；
- 交叉校验：本 cell 30个特征与上方 notebook 实现逐项 `np.allclose` 比对，口径不一致当堂暴露。

In [ ]:
def p22():
    """教师参考实现(独立可跑, 不依赖前面任何cell): 合成信号 → 30维特征 → Z-score → 断言自检
    与 notebook 主体同一口径: 单边幅值谱 ×2/N, DC 与奈奎斯特 bin 不翻倍, 频域特征统一去 DC
    """
    import numpy as np

    N, FS = 1000, 100
    t = np.arange(N) / FS

    def spectrum(x, fs=FS):
        n = x.size
        amp = np.abs(np.fft.rfft(x)) * 2.0 / n
        amp[0] *= 0.5                                     # DC 不翻倍
        if n % 2 == 0:
            amp[-1] *= 0.5                                # 奈奎斯特 bin 不翻倍
        return np.fft.rfftfreq(n, 1.0 / fs), amp

    def band(x):
        f, a = spectrum(x)
        return f[1:], a[1:]                               # 去 DC

    def wf(x):
        f, a = band(x)
        return f, a / (a.sum() + 1e-12)                   # 幅值→概率权重

    def kp(x, k):
        f, a = band(x)
        pk = np.flatnonzero((a[1:-1] > a[:-2]) & (a[1:-1] >= a[2:])) + 1
        o = pk[np.argsort(a[pk])[::-1]] if pk.size >= k else np.argsort(a)[::-1]
        return float(f[o[k - 1]])

    def mom(x, order):                                     # 标准化后取矩: 偏度/峭度同一套
        y = (x - x.mean()) / (x.std() + 1e-12)
        return y ** order if order < 3 else (y * y * (y * y if order == 4 else y))

    F = {
        "mean": lambda x: x.mean(),
        "std": lambda x: x.std(),
        "max": lambda x: np.abs(x).max(),
        "ptp": lambda x: np.ptp(x),
        "rms": lambda x: np.sqrt((x ** 2).mean()),
        "var": lambda x: x.var(),
        "abs_mean": lambda x: np.abs(x).mean(),
        "crest": lambda x: np.abs(x).max() / (np.sqrt((x ** 2).mean()) + 1e-12),
        "kurtosis": lambda x: float((mom(x, 4)).mean() - 3.0),
        "clearance": lambda x: np.abs(x).max() / (np.sqrt(np.abs(x)).mean() ** 2 + 1e-12),
        "dom_freq": lambda x: float(band(x)[0][band(x)[1].argmax()]),
        "spec_energy": lambda x: float((band(x)[1] ** 2).sum()),
        "spec_centroid": lambda x: float((wf(x)[0] * wf(x)[1]).sum()),
        "f2": lambda x: kp(x, 2),
        "f3": lambda x: kp(x, 3),
        "spec_var": lambda x: float((wf(x)[1] * (wf(x)[0] - (wf(x)[0] * wf(x)[1]).sum()) ** 2).sum()),
        "bandwidth": lambda x: float(np.sqrt(
            (wf(x)[1] * (wf(x)[0] - (wf(x)[0] * wf(x)[1]).sum()) ** 2).sum())),
        "spec_rms_freq": lambda x: float(np.sqrt((wf(x)[1] * wf(x)[0] ** 2).sum())),
        "f90_energy": lambda x: float(band(x)[0][np.searchsorted(
            np.cumsum(band(x)[1] ** 2) / ((band(x)[1] ** 2).sum() + 1e-12), 0.90)]),
        "spec_flatness": lambda x: float(np.exp(np.log(band(x)[1] ** 2 + 1e-12).mean())
                                         / (band(x)[1] ** 2 + 1e-12).mean()),
        "energy_ratio_low": lambda x: float(
            (band(x)[1][band(x)[0] <= 20] ** 2).sum() / ((band(x)[1] ** 2).sum() + 1e-12)),
        "energy_ratio_high": lambda x: float(
            (band(x)[1][band(x)[0] > 20] ** 2).sum() / ((band(x)[1] ** 2).sum() + 1e-12)),
        "dom_band_energy": lambda x: float((band(x)[1][np.abs(
            band(x)[0] - band(x)[0][band(x)[1].argmax()]) <= 5] ** 2).sum()),
        "skewness": lambda x: float(mom(x, 3).mean()),
        "cv": lambda x: float(x.std() / (np.abs(x.mean()) + 1e-12)),
        "shape_factor": lambda x: float(np.sqrt((x ** 2).mean()) / (np.abs(x).mean() + 1e-12)),
        "spec_entropy": lambda x: (lambda p: float(-(p * np.log(p + 1e-12)).sum()))(
            band(x)[1] ** 2 / ((band(x)[1] ** 2).sum() + 1e-12)),
        "impulse": lambda x: float(np.abs(x).max() / (np.abs(x).mean() + 1e-12)),
        "margin": lambda x: float(np.abs(x).max() / (
            np.partition(x, 3 * x.size // 4)[3 * x.size // 4]
            - np.partition(x, x.size // 4)[x.size // 4] + 1e-12)),
        "zcr": lambda x: float((np.diff(np.sign(x - x.mean())) != 0).mean()),
    }
    assert len(F) == 30, "参考实现应覆盖30个特征"

    # ---- 断言A: 纯正弦(A=3, 8Hz, 窗内整周期)的物理真值 ----
    sine = 3.0 * np.sin(2 * np.pi * 8 * t)
    v = {k: f(sine) for k, f in F.items()}
    assert abs(v["mean"]) < 1e-12 and abs(v["skewness"]) < 1e-9
    assert abs(v["rms"] - 3 / np.sqrt(2)) < 1e-12 and abs(v["var"] - 4.5) < 1e-12
    # 采样点未严格落在波峰(每周期12.5个点), 所以极值类指标用1%容差
    assert abs(v["max"] - 3) < 0.03 and abs(v["ptp"] - 6) < 0.06
    assert abs(v["crest"] - np.sqrt(2)) < 1e-2
    assert abs(v["kurtosis"] + 1.5) < 1e-9            # 正弦超额峭度 = -1.5
    assert abs(v["abs_mean"] - 6 / np.pi) < 1e-2
    assert abs(v["shape_factor"] - np.pi / (2 * np.sqrt(2))) < 1e-2
    assert abs(v["impulse"] - np.pi / 2) < 1e-2
    assert abs(v["dom_freq"] - 8.0) < 1e-12 and abs(v["spec_centroid"] - 8.0) < 1e-6
    assert abs(v["spec_energy"] - 9.0) < 1e-9         # = 2×var(帕塞瓦尔), 与方差互校
    assert abs(v["f90_energy"] - 8.0) < 1e-12 and v["spec_var"] < 1e-6
    assert v["energy_ratio_low"] > 0.999 and v["energy_ratio_high"] < 0.001
    assert v["spec_entropy"] < 1e-6 and v["spec_flatness"] < 0.05
    print("  断言A 纯正弦真值: crest=%.4f 峭度=%.4f 主频=%.1fHz ✓" % (v["crest"], v["kurtosis"], v["dom_freq"]))

    # ---- 断言B: 换成"磨损型"信号(33Hz冲击串+60Hz)后冲击类指标同向变大 ----
    r1 = np.random.RandomState(7)
    worn = (sine + 1.2 * np.abs(np.sin(2 * np.pi * 33 * t)) * r1.binomial(1, 0.08, N)
            + 0.4 * np.sin(2 * np.pi * 60 * t))
    w = {k: f(worn) for k, f in F.items()}
    for k in ("kurtosis", "crest", "clearance", "impulse", "margin", "zcr", "spec_energy",
              "spec_entropy", "energy_ratio_high", "bandwidth", "spec_rms_freq"):
        assert w[k] > v[k], k
    assert w["max"] > v["max"] and w["energy_ratio_low"] < v["energy_ratio_low"]
    assert w["f90_energy"] == v["f90_energy"]   # 基频仍占90%能量 → 单指标漏判(教学点)
    print("  断言B 冲击方向性: 峭度 %.2f→%.2f, 谱熵 %.3f→%.3f ✓"
          % (v["kurtosis"], w["kurtosis"], v["spec_entropy"], w["spec_entropy"]))

    # ---- 断言C: 两类合成数据跑完整流水线 + 标准化 + 冗余检测 ----
    rng = np.random.RandomState(42)
    sigs = []
    for i in range(50):
        s = 0.5 + 0.1 * np.sin(2 * np.pi * 8 * t) + rng.normal(0, 0.05, N)
        if i >= 25:
            s += 1.2 * np.abs(np.sin(2 * np.pi * 33 * t)) * rng.binomial(1, 0.08, N)
            s += 0.4 * np.sin(2 * np.pi * 60 * t)
        sigs.append(s)
    sigs = np.array(sigs)
    X = np.array([[f(s) for f in F.values()] for s in sigs])
    assert X.shape == (50, 30) and np.isfinite(X).all()
    sd = X.std(0)
    assert (sd > 0).all(), "参考实现不应出现零方差列"
    Z = (X - X.mean(0)) / np.where(sd > 0, sd, 1.0)
    assert np.abs(Z.mean(0)).max() < 1e-9 and np.abs(Z.std(0) - 1).max() < 1e-9
    C = np.corrcoef(Z.T)
    keys = list(F)
    i_lo, i_hi = keys.index("energy_ratio_low"), keys.index("energy_ratio_high")
    assert abs(C[i_lo, i_hi] + 1) < 1e-9     # 低频占比+高频占比=1 → r=-1, 现成的冗余案例
    pooled = np.sqrt((X[:25].var(0) + X[25:].var(0)) / 2.0)
    with np.errstate(divide="ignore", invalid="ignore"):
        d = np.abs(X[25:].mean(0) - X[:25].mean(0)) / pooled
    d = np.where(pooled < 1e-9, np.inf, d)
    print("  断言C 50×30流水线: 标准化+相关性OK, 最强可分特征 %s(d=%.1f) ✓"
          % (keys[int(np.argmax(d))], d.max()))

    # ---- 断言D: 与 notebook 主体的30个特征逐列交叉校验(口径必须完全一致) ----
    reg = globals().get("FEATURES")
    if reg:
        assert set(reg) == set(F), ("特征名不一致", set(reg) ^ set(F))
        for k in keys:
            fn = reg[k][0]
            a = fn(sigs[0], FS) if fn.__code__.co_argcount > 1 else fn(sigs[0])
            assert np.allclose(a, F[k](sigs[0]), rtol=1e-9, atol=1e-12), k
        print("  断言D 交叉校验: notebook 30个特征与参考实现逐项一致 ✓")
    else:
        print("  断言D 跳过: 单独运行本函数时 notebook 主体未执行")
    print("p22 参考实现: 特征矩阵 %s | 全部断言通过" % (X.shape,))
    return X, keys


X_ref, names_ref = p22()

## 运行与验证

上方参考实现可**直接运行**（合成数据、CPU 离线、固定种子 42、结果可复现），并且自带断言：
纯正弦的 crest≈1.414、rms=A/√2、峭度≈-1.5、波形因子≈1.111、主频=8Hz；加冲击后峭度/裕度/脉冲指标显著上升。

教师可对照学生版逐 cell 讲解：学生在留白处填写 → 此处给出完成态；两侧同名特征用 `np.allclose` 交叉校验，口径不一致当堂可发现。

> 多解法对比见同目录《多解法手册.md》与 `教师参考/solutions_multi*.py`。